# NB12 — Business-Aware Forecast Evaluation

## Objective

NB12 evaluates forecast quality from a business perspective.

Previous notebooks focused on:

- predictive accuracy
- model robustness
- compute cost
- benchmark quality

NB12 focuses on **operational usefulness**.

The goal is not only to identify which model predicts better,  
but which model fails in a less costly way for the business.

This notebook evaluates:

- forecast bias (under vs over)
- promo-day forecast risk
- top-selling segment exposure
- high-sales operational error
- weighted business impact

This is not a model training notebook.  
This is a business-facing forecast risk evaluation notebook.

In [8]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent))

In [9]:
# Setup
SEED = 42

DATA_DIR = Path("../data")
ARTIFACTS_DIR = Path("../artifacts/nb12_business_aware_evaluation")
NB9_DIR = Path("../artifacts/nb9_autogluon_tabular")
NB11_DIR = Path("../artifacts/nb11_autogluon_extended_budget")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

NB9_PRED_PATH = NB9_DIR / "nb9_fold_predictions.csv"
NB11_PRED_PATH = NB11_DIR / "nb11_fold_predictions.csv"

print("Setup OK")
print("Artifacts dir:", ARTIFACTS_DIR.resolve())

Setup OK
Artifacts dir: /home/donatocorbacio/projects/store-sales-project/artifacts/nb12_business_aware_evaluation


## Load Prediction Artifacts

NB12 compares the operational behavior of the two validated AutoML candidates:

- **NB9** → best default cost/performance configuration
- **NB11** → extended-budget variant with lower absolute error and higher compute cost

The goal is not to retrain models,  
but to evaluate how their forecast errors translate into operational risk.

In [10]:
nb9 = pd.read_csv(NB9_PRED_PATH, parse_dates=["date"])
nb11 = pd.read_csv(NB11_PRED_PATH, parse_dates=["date"])

print("NB9 shape:", nb9.shape)
print("NB11 shape:", nb11.shape)

display(nb9.head())

NB9 shape: (49896, 7)
NB11 shape: (49896, 7)


,fold,date,store_nbr,family,sales,prediction,onpromotion
0,2,2017-07-19,1,0,7.0,5.506095,0
1,2,2017-07-20,1,0,4.0,6.113786,0
2,2,2017-07-21,1,0,10.0,5.213129,0
3,2,2017-07-22,1,0,8.0,4.225158,0
4,2,2017-07-23,1,0,0.0,3.790067,0


## Standardize Prediction Frames

To compare both models consistently, predictions are converted into a shared evaluation format.

For each forecast we compute:

- raw forecast error
- absolute error
- relative error
- underforecast flag
- overforecast flag

This allows business-oriented error segmentation.

In [11]:
def prepare_eval_frame(df, model_name):
    out = df.copy()
    out["model"] = model_name
    out["error"] = out["sales"] - out["prediction"]
    out["abs_error"] = np.abs(out["error"])
    out["pct_error"] = np.where(out["sales"] > 0, out["error"] / out["sales"], np.nan)
    out["underforecast"] = (out["prediction"] < out["sales"]).astype(int)
    out["overforecast"] = (out["prediction"] > out["sales"]).astype(int)
    return out

nb9_eval = prepare_eval_frame(nb9, "NB9_AutoGluon_600s")
nb11_eval = prepare_eval_frame(nb11, "NB11_AutoGluon_3600s")

eval_df = pd.concat([nb9_eval, nb11_eval], ignore_index=True)

print("Combined eval shape:", eval_df.shape)

Combined eval shape: (99792, 13)


## Add Business Segments

Forecast errors are segmented using operationally relevant business slices:

- promo vs non-promo
- low vs high sales volume
- top-selling families
- weighted revenue exposure

This helps evaluate not only forecast quality,  
but also where forecast errors are most costly.

In [12]:
eval_df["promo_flag"] = (eval_df["onpromotion"] > 0).astype(int)

eval_df["sales_segment"] = pd.qcut(
    eval_df["sales"].clip(lower=0),
    q=4,
    labels=["low", "mid", "high", "very_high"]
)

top_families = (
    eval_df.groupby("family")["sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

eval_df["top_family_flag"] = eval_df["family"].isin(top_families).astype(int)

eval_df["revenue_proxy"] = eval_df["sales"]
eval_df["weighted_abs_error"] = eval_df["abs_error"] * eval_df["revenue_proxy"]

## Forecast Bias

This section evaluates directional forecast bias:

- underforecast → stockout risk
- overforecast → overstock risk

This is often more important operationally than raw average error.

In [13]:
bias_summary = (
    eval_df.groupby("model")
    .agg(
        underforecast_rate=("underforecast", "mean"),
        overforecast_rate=("overforecast", "mean"),
        mean_error=("error", "mean"),
        mean_abs_error=("abs_error", "mean"),
    )
    .reset_index()
)

display(bias_summary)

,model,underforecast_rate,overforecast_rate,mean_error,mean_abs_error
0,NB11_AutoGluon_3600s,0.434885,0.486893,3.335853,55.780798
1,NB9_AutoGluon_600s,0.417849,0.509640,5.479188,58.002139


## Promo-Day Risk

Promo periods are operationally high-risk because forecast misses
can directly impact stock, margin and campaign execution.

This section isolates forecast behavior on promo days.

In [14]:
promo_risk = (
    eval_df.groupby(["model", "promo_flag"])
    .agg(
        mae=("abs_error", "mean"),
        mean_error=("error", "mean"),
        weighted_abs_error=("weighted_abs_error", "mean"),
    )
    .reset_index()
)

display(promo_risk)

,model,promo_flag,mae,mean_error,weighted_abs_error
0,NB11_AutoGluon_3600s,0,9.673521,0.951950,4568.741330
1,NB11_AutoGluon_3600s,1,117.884655,6.546834,409180.296488
2,NB9_AutoGluon_600s,0,9.878142,-0.079068,4868.556000
3,NB9_AutoGluon_600s,1,122.822400,12.965841,441214.618642


## Sales Segment Risk

Forecast errors do not have the same cost across sales volumes.

This section evaluates whether models fail differently on:

- low-volume sales
- high-volume sales
- operationally critical segments

In [15]:
segment_risk = (
    eval_df.groupby(["model", "sales_segment"])
    .agg(
        mae=("abs_error", "mean"),
        mean_error=("error", "mean"),
        weighted_abs_error=("weighted_abs_error", "mean"),
    )
    .reset_index()
)

display(segment_risk)

,model,sales_segment,mae,mean_error,weighted_abs_error
0,NB11_AutoGluon_3600s,low,1.866153,-1.140339,3.097680
1,NB11_AutoGluon_3600s,mid,6.048371,-0.460711,93.154241
2,NB11_AutoGluon_3600s,high,23.918258,-2.684390,3537.800183
3,NB11_AutoGluon_3600s,very_high,191.660799,17.684930,704249.038050
4,NB9_AutoGluon_600s,low,1.603323,-1.112865,3.106573
5,NB9_AutoGluon_600s,mid,6.257462,-1.663363,98.956890
6,NB9_AutoGluon_600s,high,24.868030,-5.025582,3668.497316
7,NB9_AutoGluon_600s,very_high,199.690898,29.669001,759398.931797


## Top-Family Operational Exposure

Not all product families matter equally.

This section isolates forecast risk on top-selling families,
where forecast misses are typically more expensive.

In [16]:
family_risk = (
    eval_df.groupby(["model", "top_family_flag"])
    .agg(
        mae=("abs_error", "mean"),
        mean_error=("error", "mean"),
        weighted_abs_error=("weighted_abs_error", "mean"),
    )
    .reset_index()
)

display(family_risk)

,model,top_family_flag,mae,mean_error,weighted_abs_error
0,NB11_AutoGluon_3600s,0,10.564396,0.425101,2060.352036
1,NB11_AutoGluon_3600s,1,159.778524,10.030582,579256.136089
2,NB9_AutoGluon_600s,0,10.689011,-0.361204,2054.411704
3,NB9_AutoGluon_600s,1,166.822331,18.912091,624880.591304


## High-Risk Operational Slice

This section isolates the most operationally sensitive forecast slice:

- promo days
- high / very high sales

This approximates the scenarios where forecast errors are most expensive.

In [17]:
high_risk_slice = eval_df[
    (eval_df["promo_flag"] == 1) &
    (eval_df["sales_segment"].isin(["high", "very_high"]))
].copy()

high_risk_summary = (
    high_risk_slice.groupby("model")
    .agg(
        mae=("abs_error", "mean"),
        mean_error=("error", "mean"),
        weighted_abs_error=("weighted_abs_error", "mean"),
        underforecast_rate=("underforecast", "mean"),
    )
    .reset_index()
)

display(high_risk_summary)

,model,mae,mean_error,weighted_abs_error,underforecast_rate
0,NB11_AutoGluon_3600s,135.160752,8.402361,473482.360816,0.478282
1,NB9_AutoGluon_600s,140.767049,16.015377,510550.655231,0.468975


## Business Decision Table

This table summarizes which model is operationally safer
under business-relevant conditions.

In [18]:
decision_rows = []

for model in eval_df["model"].unique():
    subset = eval_df[eval_df["model"] == model].copy()

    decision_rows.append({
        "model": model,
        "overall_mae": subset["abs_error"].mean(),
        "weighted_abs_error": subset["weighted_abs_error"].mean(),
        "underforecast_rate": subset["underforecast"].mean(),
        "promo_mae": subset[subset["promo_flag"] == 1]["abs_error"].mean(),
        "high_segment_mae": subset[subset["sales_segment"].isin(["high", "very_high"])]["abs_error"].mean(),
    })

business_decision = pd.DataFrame(decision_rows)
display(business_decision)

,model,overall_mae,weighted_abs_error,underforecast_rate,promo_mae,high_segment_mae
0,NB9_AutoGluon_600s,58.002139,190789.617644,0.417849,122.822400,112.314516
1,NB11_AutoGluon_3600s,55.780798,176968.165385,0.434885,117.884655,107.823160


## Save Artifacts

In [19]:
bias_summary.to_csv(ARTIFACTS_DIR / "nb12_bias_summary.csv", index=False)
promo_risk.to_csv(ARTIFACTS_DIR / "nb12_promo_risk.csv", index=False)
segment_risk.to_csv(ARTIFACTS_DIR / "nb12_segment_risk.csv", index=False)
family_risk.to_csv(ARTIFACTS_DIR / "nb12_family_risk.csv", index=False)
high_risk_summary.to_csv(ARTIFACTS_DIR / "nb12_high_risk_summary.csv", index=False)
business_decision.to_csv(ARTIFACTS_DIR / "nb12_business_decision.csv", index=False)

print("Saved NB12 artifacts to:", ARTIFACTS_DIR.resolve())

Saved NB12 artifacts to: /home/donatocorbacio/projects/store-sales-project/artifacts/nb12_business_aware_evaluation


## Final Conclusion

NB12 evaluates forecast quality beyond standard regression metrics.

The objective is not only to identify which model is more accurate,  
but which model is operationally safer.

This notebook reframes forecast evaluation around business-facing risk:

- underforecast risk (stockout exposure)
- overforecast risk (overstock exposure)
- promo-day sensitivity
- high-sales operational error
- weighted business impact

NB12 complements NB9 and NB11 by shifting model evaluation  
from predictive quality to operational usefulness.

The best forecasting model is not only the one with the lowest error,  
but the one whose errors are least costly in production.